# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all Record Sets and fields by @id
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"  Record set name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"      - {field.name} (@id: {field.id}, data type: {field.data_type})")
    print()

# Store the record set ids for later use
record_set_ids = [rs.id for rs in metadata.record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each Record Set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Data for Record Set @id={record_set_id}")
    print("Columns:", df.columns.tolist())
    print(df.head(2))
    print('-'*60)

# Select the first record set as main for demonstration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Selected main record set: {main_record_set_id}")
    print("Sample columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Automatically select a numeric field (float/int) for demonstration
numeric_field = None
group_field = None

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to detect a numeric field
    for col in df.columns:
        # If most entries are numeric, select this column
        sample = df[col].dropna().head(10).astype(str)
        if sample.apply(lambda x: x.replace('.','',1).isdigit()).mean() > 0.8:
            numeric_field = col
            break
    # Try to find a group field
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < len(df) / 4 and df[col].dtype == object:
            group_field = col
            break

if numeric_field is not None:
    print(f"Selected numeric field: {numeric_field}")
    # Convert column to float if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Set a threshold for filtering
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group-by demonstration
    if group_field is not None and group_field in df.columns:
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field is not None:
    plt.figure(figsize=(8,6))
    sns.histplot(data=df, x=numeric_field, kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- We loaded the ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library by referencing structural elements via their `@id` fields.
- The notebook demonstrated how to extract record sets and fields, filter and normalize numeric data (e.g., protein measurements), and visualize distributions and groupings by selected attributes.
- For detailed, production analysis always review data types, validate field contents, and handle missing or erroneous entries per the context of the dataset.

> For more information, consult the [dataset's FAIR^2 page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and [mlcroissant documentation](https://mlcommons.org/croissant).
